# Fold maps from photos

So far, every fold map came from a crease pattern solved by Flat-Folder. That limits us to stages we can write down as crease patterns, and assumes perfect folds. This notebook measures fold maps from paper instead:

1. **Print a calibration sheet**: a black-and-white ChArUco board, the target OpenCV uses to calibrate cameras. It's a checkerboard with a tiny numbered marker (an ArUco code) in every white square.
2. **Fold it**, and at every stage you want a picture on, **photograph each view** straight on, against a plain colored backdrop.
3. **Recover the map**: every marker found in a photo says which part of the sheet that patch of paper is, and its corners say how it's turned. Markers on the same rigid piece of paper agree on one transform; RANSAC groups them into pieces. Then each pixel of paper goes to the piece whose transform best predicts the board's pattern there.

The result, for each photo pixel, is the point of the printed sheet it shows: the same thing `fold_map.py` computes from a crease pattern, but for **any stage of any model**, from your actual folds. That's the input the story generator needs.

**Why black and white:** a printer can't get it wrong the way it shifted colors in trial 1, a photo can be thresholded regardless of lighting, and the colored backdrop separates cleanly from paper that has no color at all.

Before printing anything, this notebook tests the pipeline on photos **simulated** from the fold maps we already know (crane, bird base, preliminary base), with blur, noise, uneven light and shadows along layer edges. The simulation knows the true map, so it measures how much of each view is recovered correctly. Runs on CPU in a few minutes, no GPU needed.

## Setup

Clones [rath-hy/folded-illusions](https://github.com/rath-hy/folded-illusions/tree/main/fold-map) on Colab, or uses the local files if you run this from that folder. Needs OpenCV 4.7 or later (Colab has it; locally, `pip install opencv-python`).

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "fold-map"
if not os.path.exists("photo_map.py"):
    if not os.path.exists("folded-illusions"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/rath-hy/folded-illusions.git"], check=True)
    os.chdir(os.path.join("folded-illusions", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "decal", "illusion", "story", "photo_map"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
import cv2

from fold_map import FoldedModel
from story import STAGES
from photo_map import (evaluate, load_photo, make_board, map_area, marked_area, on_sheet, plot_projection,
                       plot_recovery, project_picture, recover, save_board_pdf, save_print, simulate_photo,
                       straighten)
print("OpenCV", cv2.__version__)

## 1. The board

44 × 44 squares on a 200 mm square (the size the story notebooks print), so 4.5 mm squares with 3.4 mm markers. Half the squares carry a marker, 968 in all, each with its own number. That's close to the maximum: OpenCV's 4 × 4 marker set has 1,000.

Many small markers matter because a marker is useless once a crease cuts it or another layer covers part of it, and the crane's thinnest visible pieces are about a fifteenth of the sheet across. Folding scales with the paper, so what counts is squares per sheet: a bigger sheet makes bigger pieces and bigger markers alike. The bigger sheet helps elsewhere: a millimeter of folding error is a smaller share of each piece, and each marker bit is 0.57 mm, easy to print and photograph.

The PDF puts one board on A4 with a 100 mm scale bar (a copy is in [`prints/charuco_board_a4.pdf`](prints/charuco_board_a4.pdf)). **Print at 100% (not "fit to page")** and measure the bar. Cut along the thin gray line.

In [ ]:
board = make_board()           # squares=44, sheet_mm=200

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
axes[0].imshow(board.image, cmap="gray")
axes[0].set_title(f"{board.squares} x {board.squares} squares, {board.square_mm:.2f} mm each, {len(board.corners)} markers")
z = 6 * 40                     # six squares of the top-left corner
axes[1].imshow(board.image[:z, :z], cmap="gray", interpolation="nearest")
axes[1].set_title("close-up: six squares")
for ax in axes:
    ax.axis("off")
plt.show()

save_board_pdf(board, "charuco_board_a4.pdf")
print("saved charuco_board_a4.pdf")
try:
    from google.colab import files
    files.download("charuco_board_a4.pdf")
except ImportError:
    pass

## 2. A simulated photo

`simulate_photo` folds the board with a known fold map and renders a view as a camera would see it:

- `px_mm`: camera pixels per mm of paper. A phone photo of a folded crane filling the frame gets 20 or more.
- `blur`, `noise`: a slightly soft, slightly grainy photo.
- `light`: light falling off across the frame (0.3: the dimmest corner is 30% darker).
- `shadow`: dark lines where one layer's edge sits on another, as real folds have.
- The backdrop is blue, as yours should be: any plain, clearly colored surface.

In [ ]:
models = {name: FoldedModel(cp, st) for name, cp, st, _ in STAGES}

photo, truth = simulate_photo(models["crane"], "front", board, px_mm=15)
plt.figure(figsize=(8, 6.5))
plt.imshow(photo)
plt.title("simulated photo: the crane's front, folded from the board")
plt.axis("off")
plt.show()

## 3. Recover the map

From the photo alone (the truth is used only to score it):

1. **Markers**: OpenCV finds each marker and its number.
2. **Pieces**: each marker gives a transform from the sheet to the photo. The transform most markers agree with is a rigid piece; refit it to all of them, set them aside, repeat.
3. **Every pixel**: each piece's transform predicts the whole board's pattern across the photo. Each paper pixel goes to the piece whose prediction matches best in a window about one square wide (normalized cross-correlation, so uneven light doesn't matter).

The third panel is the visibility map: where each piece's paper sits on the printed sheet, the thing you'd otherwise shade by hand and transcribe.

In [ ]:
result = recover(photo, board)
plot_recovery(photo, result, board, truth)
plt.show()
ev = evaluate(result, truth)
print(f"{ev['markers']} markers, {ev['pieces']} pieces: {ev['accurate']:.1%} of the view within 0.5 mm, "
      f"{ev['wrong']:.1%} off by over 1 mm, {ev['missing']:.1%} not recovered")

**What's missing:** the neck and tail tips, which are too thin to hold a whole marker, and thin seams where two pieces meet (the matching window straddles both). The wrong pixels sit on those seams too. Fixes for later:

- **Borrow from the crease pattern when there is one.** A sliver's transform is its neighbor's reflected across their shared crease.
- **Match texture directly:** a sliver with no marker could be found by searching the sheet for its pattern.
- **Print with a bleed** anyway, as the decals do, since seams are where folding errors land too.

### Straighten the seams

In a flat fold, every boundary between visible pieces is a straight line: a crease, a sheet edge or the edge of a layer on top. Rigid folding keeps those edges straight, and so does a camera. The ragged seams above come from the matching window, which straddles two pieces near a boundary.

`straighten` uses that fact:

1. **Drop specks:** fragments of a piece smaller than about one square.
2. **Fit lines:** for each pair of neighboring pieces, find straight runs along their boundary with RANSAC. A boundary with a bend gets several lines.
3. **Redraw:** pixels of those two pieces near a line go to its correct side.
4. **Fill narrow gaps:** an unmatched pixel near a piece joins it only if the piece's predicted pattern matches the photo there. Without that check, pieces spread into paper they don't own, like the base of the crane's head.

It can't recover paper that has no transform, like the head and tail tips with no whole marker. The fitted lines also aren't creases you could compute from the two transforms: a crease flips the paper, so two pieces showing the same side of a flat fold are never joined by one crease. Their transforms disagree all along the seam.

In [ ]:
straight = straighten(result, photo, board)
plot_recovery(photo, straight, board, truth)
plt.show()
for name, r in (("before", result), ("after", straight)):
    ev = evaluate(r, truth)
    print(f"{name:<7}{ev['accurate']:6.1%} within 0.5 mm, {ev['wrong']:.2%} off by over 1 mm, {ev['missing']:.2%} not recovered")
print(f"{len(straight['seams'])} straight seams")

Scored against the simulation's true map, before and after, on models and views the method was not developed on (7 other Flat-Folder models, 11 views):

| | within 0.5 mm | off by over 1 mm | not recovered |
|---|---|---|---|
| before | 96.5% | 1.17% | 2.37% |
| after | 97.6% | 0.61% | 1.82% |

On 21 runs of the three stages, across seeds, resolutions, harsh photos and camera tilts, it's a smaller gain: 95.1% to 95.8% within 0.5 mm, 1.02% to 0.78% wrong, 3.91% to 3.44% missing. It's a small numerical gain and a large visual one, since the errors it removes sit on the edges, where a print shows them. Real paper may not gain as much: curved or bulging edges aren't straight, and this hasn't been tested on a real multi-piece fold yet.

## 4. How robust is it?

Each table changes one thing from the run above. This takes a few minutes.

In [ ]:
def score(name, view, board, **kw):
    photo, truth = simulate_photo(models[name], view, board, **kw)
    return evaluate(recover(photo, board), truth)

def row(label, ev):
    print(f"  {label:<44} markers {ev['markers']:>4}   pieces {ev['pieces']:>2}   within 0.5 mm {ev['accurate']:6.1%}"
          f"   off > 1 mm {ev['wrong']:5.1%}   missing {ev['missing']:5.1%}")

print("Square size (crane front, 15 px/mm)")
for n in (24, 32, 40, 44):
    row(f"{n} squares, {board.sheet_mm / n:.1f} mm", score("crane", "front", make_board(n), px_mm=15))

print("\nCamera resolution (crane front)")
for px in (4, 6, 8, 15, 25):
    row(f"{px} px/mm", score("crane", "front", board, px_mm=px))

print("\nHarsher photos (crane front, 15 px/mm)")
for kw in (dict(blur=2.0), dict(noise=0.06), dict(light=0.6), dict(shadow=0.9),
           dict(blur=2.0, noise=0.05, light=0.5, shadow=0.8)):
    row(", ".join(f"{k} {v}" for k, v in kw.items()), score("crane", "front", board, px_mm=15, **kw))

print("\nEvery stage (15 px/mm)")
for name in models:
    for view in ("front", "back"):
        row(f"{name}, {view}", score(name, view, board, px_mm=15))

What the tables say (from a run on a laptop):

- **Squares per sheet dominate.** With 24 squares a side, 44% of the crane is lost, because only 17 markers survive the folds. With 44, under 4% is lost. 44 is the most the 4 × 4 marker set allows; going further would need a larger set (5 × 5 markers, which need more room per bit) or two boards with different sets side by side.
- **6 pixels per mm of paper is enough** (about 3.4 pixels per marker bit); at 4, most markers are unreadable. A phone held so the folded crane fills the frame gives 20 or more.
- **Blur, noise, light and shadows barely matter,** even all together (95.1% within 0.5 mm becomes 94.4%).
- **The bases are easy** (99% within 0.5 mm): their pieces are big. The piece counts match the geometry we know: 1 for the preliminary base, 2 for the bird base (one view also finds a tiny extra piece on a seam).

The simulation is kinder than reality in two ways: its folds are perfect and flat (real paper bulges, and layers sit at slightly different heights), and the camera looks straight down (a tilted photo needs `recover(photo, board, kind="homography")`). So the next test is real paper.

## 5. Your photos

Print the board, fold it, and photograph each view you want a picture on.

- **Backdrop:** a plain, clearly colored surface (blue or green paper, a cutting mat). Not white, black or gray: the board is all three.
- **Straight on,** from far enough that the view fills most of the frame without wide-angle distortion. Phone at about 30 cm, zoomed 2x, works well.
- **Even light,** no flash. Daylight from a window is fine. Hard shadows across the paper confuse the matching.
- **Flatten it gently** under a sheet of glass or clear plastic if layers spring up.

Then upload the photo here (or set `PHOTO` to its path). The map is saved as `.npz`: for each photo pixel, the sheet point in mm (NaN where no piece matched), and which piece.

In [ ]:
PHOTO = None                   # e.g. "crane_front.jpg"; None asks for an upload on Colab

if PHOTO is None and "google.colab" in sys.modules:
    from google.colab import files
    PHOTO = next(iter(files.upload()))
if PHOTO:
    img = cv2.cvtColor(cv2.imread(PHOTO), cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    scale = 2000 / max(img.shape[:2])              # about 2000 pixels across is plenty
    if scale < 1:
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    mine = recover(img, board)                     # kind="homography" if the photo is at an angle
    plot_recovery(img, mine, board)
    plt.show()
    name = os.path.splitext(os.path.basename(PHOTO))[0] + "_map.npz"
    np.savez_compressed(name, sheet_mm=mine["sheet_mm"].astype(np.float32), label=mine["label"].astype(np.int16))
    print(f"{len(mine['markers'])} markers, {len(mine['pieces'])} pieces; saved {name}")

## 6. Put a picture on your fold

The Ambassadors idea: a photo records what the viewer sees from one spot, so it can tell us what to print for that spot. For each sheet point visible in a chosen area of the photo, print the picture's color at the place that point lands in the photo. Fold the print the same way, look from the same spot, and the picture lines up.

You need three files:

1. **The photo** of the folded board, from where the picture should be seen.
2. **A copy of the photo with the area marked** in magenta (255, 0, 255), outlined or filled in. Don't crop or resize it, and keep any outline closed.
3. **The picture.**

Set `FIT` to `"contain"` to show the whole picture inside the area, or `"cover"` to fill the area and crop what sticks out. The picture is upright in the photo's frame.

The output is an A4 PDF of the sheet to fold. **Print at 100%, check the 100 mm bar, and cut along the black line.** The hatched paper is waste.

This section doesn't need sections 2 to 5. It uses the full-size photo and a homography for each piece, so the camera can be at an angle. Unlike section 5, it doesn't need a colored backdrop, because you marked the area yourself. Pixels in the area that match no piece, usually from glare or curved paper, take the transform of the nearest piece that did match. The first plot says how much of the area matched directly.

In [ ]:
UNMARKED = "charuco pics/box_unmarked.JPG"
MARKED = "charuco pics/box_marked.JPG"
PICTURE = "charuco pics/holbein.jpg"
FIT = "contain"                # or "cover"
OUTPUT = "prints/box_holbein_a4.pdf"

def need(path, what):
    if os.path.exists(path):
        return path
    if "google.colab" in sys.modules:
        from google.colab import files
        print(f"Upload {what}:")
        return next(iter(files.upload()))
    raise FileNotFoundError(path)

board = make_board()           # the board you folded: same squares and size
photo = load_photo(need(UNMARKED, "the photo"))
marked = load_photo(need(MARKED, "the photo with the area marked"))
picture = load_photo(need(PICTURE, "the picture"))
if photo.shape != marked.shape:
    raise ValueError(f"the two photos differ in size ({photo.shape[:2]} and {marked.shape[:2]}): don't crop the marked one")

area = marked_area(marked)
mapped = map_area(photo, board, area)
sheet, printed = project_picture(picture, mapped, board, FIT)

plot_projection(photo, mapped, sheet, board)
plt.show()

rows, cols = np.nonzero(printed)
mm = board.sheet_mm / sheet.shape[0]
print(f"{mapped['matched']:.0%} of the area matched directly, using "
      f"{len(np.unique(mapped['label'][area]))} piece(s) of paper")
print(f"the picture is printed in x {cols.min() * mm:.0f}-{cols.max() * mm:.0f} mm, "
      f"y {rows.min() * mm:.0f}-{rows.max() * mm:.0f} mm of the {board.sheet_mm:g} mm sheet")

os.makedirs(os.path.dirname(OUTPUT) or ".", exist_ok=True)
save_print(sheet, OUTPUT, board)
print("saved", OUTPUT)
try:
    from google.colab import files
    files.download(OUTPUT)
except ImportError:
    pass

### Inside the projection

The cell above runs everything in one go. This one opens it up, one step at a time, on the same photo:

1. **Markers.** Every marker found, with its number. Numbers run left to right and top to bottom across the sheet, so they already say where each patch of paper came from.
2. **Pieces.** Markers that agree on one homography share a color. Pieces with fewer than 4 markers aren't used.
3. **Fit error.** Each marker's corners against where its piece's homography predicts them. Printed below with the matrices.
4. **Rectified.** The photo warped through the marked area's main piece back onto the sheet. If the homography is right, the squares come out straight and even, and the markers land on the board's grid (drawn in cyan).
5. **Matching.** Each pixel of the area by how well the board's predicted pattern matches the photo there (local normalized cross-correlation). Pixels below 0.5 get no match on their own.
6. **Labels.** Which piece each pixel of the area went to, with pixels that had to borrow from the nearest piece in red.

In [ ]:
from photo_map import _local_ncc, _project, _px_per_mm, dense_map, piece_colors, sample_board

markers, pieces = mapped["markers"], mapped["pieces"]
x, y, w, h = cv2.boundingRect(area.astype(np.uint8))
pad = max(w, h) // 3
crop = (slice(max(y - pad, 0), y + h + pad), slice(max(x - pad, 0), x + w + pad))
extent = (crop[1].start, crop[1].stop, crop[0].stop, crop[0].start)
colors = piece_colors(len(pieces))
main = int(np.bincount(mapped["label"][area]).argmax())
M, ids = pieces[main]

# How well each homography fits its own markers.
print(f"{len(markers)} markers; {len(pieces)} pieces with at least 4 markers")
for k, (Mk, idk) in enumerate(pieces):
    P = np.concatenate([board.corners[j] for j in idk])
    Q = np.concatenate([markers[j] for j in idk])
    err = np.linalg.norm(_project(Mk, P) - Q, axis=1)
    scale = _px_per_mm(Mk, board, idk)
    star = " (the marked area)" if k == main else ""
    print(f"  piece {k}{star}: {len(idk)} markers, ids {min(idk)}-{max(idk)}, {scale:.1f} px/mm, "
          f"fit error rms {np.sqrt((err ** 2).mean()):.2f} px ({np.sqrt((err ** 2).mean()) / scale:.2f} mm), max {err.max():.2f} px")
print("\nThe marked area's homography, sheet mm -> photo pixels:")
print(np.array2string(M / M[2, 2], precision=5, suppress_small=True))

fig, axes = plt.subplots(2, 3, figsize=(18, 14))
axes = axes.ravel()

axes[0].imshow(photo)
for i, c in markers.items():
    axes[0].add_patch(plt.Polygon(c, fill=False, ec="#f33", lw=0.8))
    axes[0].text(*c[0], str(i), color="#3f3", fontsize=6)
axes[0].set_title("1. markers and their numbers")

axes[1].imshow(photo * 0.6 + 0.4)
for k, (_, idk) in enumerate(pieces):
    for j in idk:
        axes[1].add_patch(plt.Polygon(markers[j], color=colors[k]))
axes[1].set_title("2. markers by piece (one homography each)")

used = set(j for _, idk in pieces for j in idk)
for i, c in markers.items():
    if i not in used:
        axes[1].add_patch(plt.Polygon(c, fill=False, ec="k", lw=0.5))

axes[2].imshow(photo[crop], extent=extent)
for j in ids:
    pred = _project(M, board.corners[j])
    axes[2].add_patch(plt.Polygon(markers[j], fill=False, ec="#f33", lw=1))
    axes[2].add_patch(plt.Polygon(pred, fill=False, ec="#0ff", lw=1))
axes[2].set_xlim(extent[0], extent[1]); axes[2].set_ylim(extent[2], extent[3])
axes[2].set_title("3. found (red) and predicted by the homography (cyan)")

# Rectify: for a patch of the sheet, look up the photo through the homography.
rows, cols = np.nonzero(area)
corners = _project(np.linalg.inv(M), np.c_[cols, rows].astype(float))
lo, hi = np.percentile(corners, [0, 100], axis=0)
lo, hi = np.maximum(lo - 8, 0), np.minimum(hi + 8, board.sheet_mm)
s = 12                         # pixels per mm
xs, ys = np.arange(lo[0], hi[0], 1 / s), np.arange(lo[1], hi[1], 1 / s)
gx, gy = np.meshgrid(xs, ys)
p = _project(M, np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape + (2,)).astype(np.float32)
rectified = cv2.remap(photo, p[..., 0], p[..., 1], cv2.INTER_LINEAR)
axes[3].imshow(rectified, extent=(lo[0], xs[-1], ys[-1], lo[1]))
grid = np.arange(0, board.squares + 1) * board.square_mm
for g in grid:
    axes[3].axvline(g, color="#0ff", lw=0.4); axes[3].axhline(g, color="#0ff", lw=0.4)
axes[3].set_xlim(lo[0], xs[-1]); axes[3].set_ylim(ys[-1], lo[1])
axes[3].set_xlabel("sheet x (mm)"); axes[3].set_ylabel("sheet y (mm)")
axes[3].set_title("4. rectified onto the sheet, with the board's grid")

# The matching score that dense_map computes, for the main piece.
H, W = area.shape
yy, xx = np.mgrid[crop[0].start:crop[0].stop, crop[1].start:crop[1].stop]
xy = _project(np.linalg.inv(M), np.c_[xx.ravel(), yy.ravel()].astype(float)).reshape(xx.shape + (2,))
pred = sample_board(board, xy)
valid = area[crop] & np.isfinite(pred)
pred = cv2.GaussianBlur(np.nan_to_num(pred, nan=0.5).astype(np.float32), (0, 0), 1.0)
win = int(round(board.square_mm * _px_per_mm(M, board, ids))) | 1
ncc = _local_ncc(photo[crop].mean(-1), pred, valid, win)
im = axes[4].imshow(np.where(area[crop], ncc, np.nan), extent=extent, vmin=-1, vmax=1, cmap="RdYlGn")
fig.colorbar(im, ax=axes[4], fraction=0.04)
axes[4].set_title("5. match with the predicted board (below 0.5: no match)")

_, direct = dense_map(photo, board, pieces, mask=area)
labels = np.ones(area.shape + (3,))
labels[mapped["label"] >= 0] = colors[mapped["label"][mapped["label"] >= 0]]
labels[area & (direct < 0)] = (1, 0, 0)
axes[5].imshow(labels[crop], extent=extent)
axes[5].set_title("6. piece per pixel; red borrowed from the nearest piece")

all_corners = np.concatenate(list(markers.values()))
(x0, y0), (x1, y1) = all_corners.min(0) - 60, all_corners.max(0) + 60
for ax in axes[:2]:            # panels 1 and 2 show all the paper
    ax.set_xlim(x0, x1); ax.set_ylim(y1, y0)
for i in (0, 1, 2, 4, 5):
    axes[i].axis("off")
fig.tight_layout()
plt.show()

**Folding the print.** It only works if the print takes the same shape as the board:

- **Same orientation.** The sheet has no markers now, so use the picture to line it up: compare it with the marked area of your photo, and fold so it ends up the same way round in the model.
- **Same shape and viewpoint.** Partly open folds need to open to the same angle, and the camera has to return to the same place. Use your unfolded board's creases as a guide for where to fold.
- **Check it.** The third plot is the prediction. It only repaints the marked area, and shows the board everywhere else. On the print, that paper is white.

## Next

- **Feed the maps to the story generator.** The generator wants each view as a canvas of sheet pixels (`FoldViews`). A recovered map converts directly: resample the photo so one canvas pixel is about one sheet pixel, and look up the sheet pixel each canvas pixel shows.
- **Measure folding consistency.** Fold two boards the same way and compare their maps. The disagreement is how much bleed a print needs, and how precise the pictures can be.
- **Recover slivers** with the crease pattern or texture matching (above).
- **Test `straighten` on real paper,** on a multi-piece fold.
- **Non-flat views.** Nothing here assumes the model is flat: a crane with spread wings, photographed from a chosen angle, gives a map too. Each piece then needs a homography rather than a rotation and shift, and curved paper would need a finer model.

## References

- S. Garrido-Jurado et al. [Automatic generation and detection of highly reliable fiducial markers under occlusion](https://doi.org/10.1016/j.patcog.2014.01.005) (ArUco). *Pattern Recognition*, 2014.
- OpenCV. [Detection of ChArUco boards](https://docs.opencv.org/4.x/df/d4a/tutorial_charuco_detection.html).
- M. A. Fischler, R. C. Bolles. [Random Sample Consensus](https://doi.org/10.1145/358669.358692) (RANSAC). *Communications of the ACM*, 1981.